In [ ]:
!pip install opencv-python

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!nvidia-smi

Thu Apr 10 15:52:37 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   37C    P8             11W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# !pip install torch==2.0.1+cu118 torchvision==0.15.2+cu118 torchaudio==2.0.2+cu118 --index-url https://download.pytorch.org/whl/cu118

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset
from torchvision.models.segmentation import deeplabv3_mobilenet_v3_large
from PIL import Image
import numpy as np
import sys
from tqdm import tqdm
import time
import matplotlib.pyplot as plt
import cv2


In [ ]:
class SegmentationDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.images = sorted(os.listdir(image_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir, self.images[idx].replace('.jpg', '_mask.png'))

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)

        mask = (mask > 0).float()  
        return image, mask


In [ ]:
transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor()
])

train_image_path = '/content/drive/MyDrive/PR/split_dataset/train/images'
train_mask_path = '/content/drive/MyDrive/PR/split_dataset/train/masks'

val_image_path = '/content/drive/MyDrive/PR/split_dataset/val/images'
val_mask_path = '/content/drive/MyDrive/PR/split_dataset/val/masks'


test_image_path = '/content/drive/MyDrive/PR/split_dataset/test/images'
test_mask_path = '/content/drive/MyDrive/PR/split_dataset/test/masks'


print(len(os.listdir(train_image_path)))
print(len(os.listdir(train_mask_path)))

print(len(os.listdir(val_image_path)))
print(len(os.listdir(val_mask_path)))

print(len(os.listdir(test_image_path)))
print(len(os.listdir(test_mask_path)))

train_dataset = SegmentationDataset(train_image_path, train_mask_path, transform)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

val_dataset = SegmentationDataset(val_image_path, val_mask_path, transform)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2)


719
719
154
154
155
155


In [ ]:
model = deeplabv3_mobilenet_v3_large(pretrained=True)
model.classifier[4] = nn.Conv2d(256, 1, kernel_size=1)
model = model.cuda()


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DeepLabV3_MobileNet_V3_Large_Weights.COCO_WITH_VOC_LABELS_V1`. You can also use `weights=DeepLabV3_MobileNet_V3_Large_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/deeplabv3_mobilenet_v3_large-fc3c493d.pth" to /root/.cache/torch/hub/checkpoints/deeplabv3_mobilenet_v3_large-fc3c493d.pth
100%|██████████| 42.3M/42.3M [00:00<00:00, 145MB/s]


In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
def compute_iou(preds, targets, threshold=0.5):
    preds = torch.sigmoid(preds)
    preds = (preds > threshold).float()

    intersection = (preds * targets).sum(dim=(1, 2, 3))
    union = ((preds + targets) >= 1).float().sum(dim=(1, 2, 3))
    iou = (intersection + 1e-6) / (union + 1e-6)
    return iou.mean().item()


In [ ]:
os.chdir('/content/drive/MyDrive/PR/')
os.getcwd()

'/content/drive/MyDrive/PR'

In [ ]:
def evaluate(model, val_loader):
    model.eval()
    total_val_loss = 0
    total_val_iou = 0

    val_loop = tqdm(val_loader, desc="Validating", unit="batch")

    with torch.no_grad():
        for images, masks in val_loop:
            images, masks = images.cuda(), masks.cuda()
            outputs = model(images)['out']
            val_loss = criterion(outputs, masks)
            val_iou = compute_iou(outputs, masks)

            total_val_loss += val_loss.item()
            total_val_iou += val_iou

            val_loop.set_postfix({
                "ValLoss": f"{val_loss.item():.4f}",
                "ValIoU": f"{val_iou:.4f}"
            })

    avg_val_loss = total_val_loss / len(val_loader)
    avg_val_iou = total_val_iou / len(val_loader)

    model.train()  
    return avg_val_loss, avg_val_iou

In [ ]:
len(val_loader)

154

In [ ]:
best_iou = 0.0
num_epochs = 10

model.train()

for epoch in range(num_epochs):
    total_train_loss = 0
    total_train_iou = 0
    epoch_start = time.time()

    print(f"\n Epoch {epoch+1}/{num_epochs}")
    loop = tqdm(train_loader, desc="Training", unit="batch")

    for images, masks in loop:
        images, masks = images.cuda(), masks.cuda()
        outputs = model(images)['out']
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_iou = compute_iou(outputs, masks)
        total_train_loss += loss.item()
        total_train_iou += batch_iou

        loop.set_postfix({
            "TrainLoss": f"{loss.item():.4f}",
            "TrainIoU": f"{batch_iou:.4f}"
        })

    avg_train_loss = total_train_loss / len(train_loader)
    avg_train_iou = total_train_iou / len(train_loader)

    avg_val_loss, avg_val_iou = evaluate(model, val_loader)

    epoch_time = time.time() - epoch_start
    print(f"\n Epoch {epoch+1} Summary:")
    print(f" Train   — Loss: {avg_train_loss:.4f} | IoU: {avg_train_iou:.4f}")
    print(f" Val     — Loss: {avg_val_loss:.4f} | IoU: {avg_val_iou:.4f}")
    print(f"⏱ Time: {epoch_time:.2f} sec")

    if avg_val_iou > best_iou:
        best_iou = avg_val_iou
        model_filename = f"/content/drive/MyDrive/PR/segmentation_models/best_model_epoch_{epoch+1:02d}_iou_{avg_val_iou:.4f}_loss_{avg_val_loss:.4f}.pth"
        torch.save(model.state_dict(), model_filename)
        print(f" Best model saved as: {model_filename}")


 Epoch 1/10


Validating: 100%|██████████| 154/154 [00:31<00:00,  4.94batch/s, ValLoss=0.0409, ValIoU=0.9707]



 Epoch 1 Summary:
 Train   — Loss: 0.0784 | IoU: 0.9386
 Val     — Loss: 0.0552 | IoU: 0.9523
⏱ Time: 188.93 sec
 Best model saved as: /content/drive/MyDrive/PR/segmentation_models/best_model_epoch_01_iou_0.9523_loss_0.0552.pth

 Epoch 2/10


Validating: 100%|██████████| 154/154 [00:02<00:00, 63.99batch/s, ValLoss=0.0253, ValIoU=0.9748]



 Epoch 2 Summary:
 Train   — Loss: 0.0490 | IoU: 0.9558
 Val     — Loss: 0.0426 | IoU: 0.9599
⏱ Time: 20.08 sec
 Best model saved as: /content/drive/MyDrive/PR/segmentation_models/best_model_epoch_02_iou_0.9599_loss_0.0426.pth

 Epoch 3/10


...


 Epoch 8 Summary:
 Train   — Loss: 0.0238 | IoU: 0.9733
 Val     — Loss: 0.0303 | IoU: 0.9675
⏱ Time: 20.15 sec
 Best model saved as: /content/drive/MyDrive/PR/segmentation_models/best_model_epoch_08_iou_0.9675_loss_0.0303.pth

 Epoch 9/10


Validating: 100%|██████████| 154/154 [00:02<00:00, 63.30batch/s, ValLoss=0.0191, ValIoU=0.9756]



 Epoch 9 Summary:
 Train   — Loss: 0.0227 | IoU: 0.9743
 Val     — Loss: 0.0289 | IoU: 0.9682
⏱ Time: 20.21 sec
 Best model saved as: /content/drive/MyDrive/PR/segmentation_models/best_model_epoch_09_iou_0.9682_loss_0.0289.pth

 Epoch 10/10


Validating: 100%|██████████| 154/154 [00:02<00:00, 62.43batch/s, ValLoss=0.0152, ValIoU=0.9811]


 Epoch 10 Summary:
 Train   — Loss: 0.0211 | IoU: 0.9758
 Val     — Loss: 0.0307 | IoU: 0.9672
⏱ Time: 20.30 sec
